# Preprocessing: Country Facts

In [8]:
import pandas as pd

In [9]:
country_facts = pd.read_csv("Country Data Original\P_Data_Extract_From_World_Development_Indicators.csv")

In [10]:
country_facts.head()

,Country Name,Country Code,Year,Time Code,"Population, total [SP.POP.TOTL]",Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST],Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST],"Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]",Children out of school (% of primary school age) [SE.PRM.UNER.ZS],GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD],GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD],Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS],Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS],Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC],Cereal yield (kg per hectare) [AG.YLD.CREL.KG],Permanent cropland (% of land area) [AG.LND.CROP.ZS],"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]","Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]"
0,Chad,TCD,1990,YR1990,5982833,..,..,16.12030029,..,..,602.6706858,2.189049847,1.38028772,2507.173441,559.2,0.021442186,11.30055905,..
1,Chad,TCD,2000,YR2000,8512093,-1.3036368,-1.0929775,21.8164196,50.3504982,..,530.6003313,1.148431445,0.999606076,1762.198792,531.3,0.023824651,5.79556105,..
2,Chad,TCD,2016,YR2016,15114655,-1.5029281,-1.4004476,40.37891281,30.1148001,867.153439,923.5658721,0.838978455,2.31170355,992.4143158,844.7,0.02930432,15.87014498,36000
3,Chad,TCD,2017,YR2017,15622759,-1.4572612,-1.3185396,36.39827728,35.80696899,840.3703197,879.0758955,0.71756175,1.65961164,960.1377068,825.2,0.02930432,15.61345027,5800
4,Chad,TCD,2018,YR2018,16156531,-1.4604181,-1.3574546,38.780691,30.18386299,892.0050404,898.834794,0.655085839,1.65270268,928.4171212,893,0.029621982,16.50983653,..


In [17]:
unique_countries = country_facts["Country Name"].unique()
print("Countries: ")
print(unique_countries)

metric_columns = country_facts.columns.difference(["Country Name", "Country_Code", "Year", "Time Code"])
print("Metric Columns:")
metric_columns

Countries: 
['Chad' 'Central African Republic' 'Kenya' 'Uganda' 'Sudan' 'South Sudan'
 'Somalia' 'Ethiopia' 'Libya' 'Egypt']
Metric Columns:


Index(['Armed forces personnel (% of total labor force) [MS.MIL.TOTL.TF.ZS]',
       'Cereal yield (kg per hectare) [AG.YLD.CREL.KG]',
       'Children out of school (% of primary school age) [SE.PRM.UNER.ZS]',
       'Control of Corruption - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_CC_EST]',
       'Country Code', 'GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]',
       'GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]',
       'Internally displaced persons, new displacement associated with conflict and violence (number of cases) [VC.IDP.NWCV]',
       'Military expenditure (% of GDP) [MS.MIL.XPND.GD.ZS]',
       'Permanent cropland (% of land area) [AG.LND.CROP.ZS]',
       'Population, total [SP.POP.TOTL]',
       'Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]',
       'Renewable internal freshwater resources per capita (cubic meters) [ER.H2O.INTR.PC]',
       'Rule of Law - Governance estimate (approx. -2.5 to +2.5) [GOV_WGI_RL_EST]',


In [15]:
import pandas as pd
import numpy as np

# Clean copy
df = country_facts.copy()

# Identify countries and metric columns
unique_countries = df["Country Name"].unique()
metric_columns = df.columns.difference(["Country Name", "Country_Code", "Year", "Time Code"])

# Replace ".." with NaN
df = df.replace("..", np.nan)
df

# Convert metric columns to numeric
df[metric_columns] = df[metric_columns].apply(pd.to_numeric, errors="coerce")

# Sort by country + year (important!)
df = df.sort_values(["Country Name", "Year"])

# Step 1: forward-fill and backward-fill within each country
df[metric_columns] = df.groupby("Country Name")[metric_columns].ffill()
df[metric_columns] = df.groupby("Country Name")[metric_columns].bfill()

# Step 2: For countries with NO data at all for a metric, impute using other countries
for metric in metric_columns:
    # Countries where the metric is still entirely missing
    missing_countries = df.groupby("Country Name")[metric].apply(lambda x: x.isna().all())
    missing_countries = missing_countries[missing_countries].index.tolist()
   
    if missing_countries:
        print("Metric: ", metric, " Mising Countries: ", missing_countries)
        
        # Compute global mean (or regional mean if you prefer)
        global_mean = df[metric].mean()

        # Impute for each missing country
        for country in missing_countries:
            df.loc[df["Country Name"] == country, metric] = global_mean

#Add missingness flags so models know it was imputed. 
for metric in metric_columns:
    df[f"{metric}_missing_flag"] = country_facts[metric].replace("..", np.nan).isna().astype(int)

print("Missing values handled successfully.")

print(df)

Metric:  Children out of school (% of primary school age) [SE.PRM.UNER.ZS]  Mising Countries:  ['Libya', 'Somalia', 'Uganda']
Metric:  Country Code  Mising Countries:  ['Central African Republic', 'Chad', 'Egypt', 'Ethiopia', 'Kenya', 'Libya', 'Somalia', 'South Sudan', 'Sudan', 'Uganda']
Metric:  GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD]  Mising Countries:  ['South Sudan']
Metric:  GNI per capita (constant 2015 US$) [NY.GNP.PCAP.KD]  Mising Countries:  ['South Sudan', 'Sudan']
Metric:  Primary completion rate, total (% of relevant age group) [SE.PRM.CMPT.ZS]  Mising Countries:  ['Libya', 'Somalia', 'South Sudan']
Metric:  Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal) [ER.GDP.FWTL.M3.KD]  Mising Countries:  ['South Sudan']
Missing values handled successfully.
                Country Name  Country Code  Year Time Code  \
12  Central African Republic           NaN  1990    YR1990   
13  Central African Republic           NaN  20

In [13]:
print(df)

                Country Name  Country Code  Year Time Code  \
12  Central African Republic           NaN  1990    YR1990   
13  Central African Republic           NaN  2000    YR2000   
14  Central African Republic           NaN  2016    YR2016   
15  Central African Republic           NaN  2017    YR2017   
16  Central African Republic           NaN  2018    YR2018   
..                       ...           ...   ...       ...   
43                    Uganda           NaN  2021    YR2021   
44                    Uganda           NaN  2022    YR2022   
45                    Uganda           NaN  2023    YR2023   
46                    Uganda           NaN  2024    YR2024   
47                    Uganda           NaN  2025    YR2025   

    Population, total [SP.POP.TOTL]  \
12                          2871910   
13                          3833416   
14                          4713663   
15                          4793511   
16                          4878657   
..                   

In [18]:
df.to_csv('Country Data Preprocessed\Country Facts.csv', index=False)